# Image Forgery Detection and Localization — End-to-End (Topic 2)

One self-contained notebook for the whole pipeline (no earlier notebook required):

`environment → CASIA v2 + manifests → train U-Net–ResNet34 (RGB / Noise / DCT) + SegFormer → calibrate the Real/Fake threshold on validation → CASIA test → Coverage (cross-dataset) → result tables → prediction → extra analyses (score methods, robustness, Columbia) → web demo`

**Usage:** `Runtime → Change runtime type → GPU`, edit the **CONFIG** cell, then `Runtime → Run all`.

- All checkpoints and results are stored on Google Drive. After a disconnect, simply *Run all* again: finished runs are skipped and unfinished runs resume from `last.pt`.
- **No GPU available?** Set `EVAL_ONLY = True`: training is skipped, the saved `thresholds.json` files are reused, and sections 5–10 run on a CPU runtime.
- The nine existing U-Net checkpoints in `checkpoints/controlled_multiseed_v1` are **reused** when they match the epoch budget.
- The test set is used only after checkpoints and thresholds have been fixed on validation.

## 0. CONFIG — the only cell you need to edit

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ImageForgeryResearch')
REPO_URL   = 'https://github.com/ngocan-dev/image-forgery-localization.git'
REPO       = Path('/content/image-forgery-localization')

# CASIA v2 data
CASIA_ZIP        = DRIVE_ROOT / 'datasets/CASIA2.zip'          # fallback when KaggleHub is unavailable
KAGGLE_DATASET   = 'chongtrung/casia-v2'
LOCAL_CASIA      = Path('/content/datasets/CASIA2')
MANIFEST_BACKUP  = DRIVE_ROOT / 'manifests/baseline_v3'        # train.csv, val.csv, test.csv
MANIFEST_PREFIX  = '/kaggle/input/casia-v2/CASIA2/'            # path prefix stored in the original manifests
LOCAL_MANIFEST   = Path('/content/manifests/casia2')

# COVERAGE (cross-dataset). The GitHub repository no longer contains the data: download the full folder from the
# authors' OneDrive link in https://github.com/wenbihan/coverage and upload the ZIP to this path.
COVERAGE_ZIP     = DRIVE_ROOT / 'datasets/COVERAGE.zip'

# Experiments
SEEDS            = [7, 21, 42]
UNET_MODES       = ['rgb', 'noise', 'frequency']   # RGB vs RGB+Noise vs RGB+DCT
SEGFORMER_ARCH   = 'segformer_mit_b2'               # Transformer upgrade ('segformer_mit_b0' for small GPUs)
SEGFORMER_MODES  = ['rgb']                          # add 'noise'/'frequency' if time allows
# 10 = reuse the nine completed 10-epoch U-Net checkpoints and train SegFormer only.
# Larger (e.g. 30) = longer-training study: retrains EVERY configuration with the same budget in separate folders.
EPOCHS           = 10

# Checkpoints and results (separated by epoch budget so two protocols are never mixed)
LEGACY_UNET_ROOT = DRIVE_ROOT / 'checkpoints/controlled_multiseed_v1'   # {mode}_seed{seed}/ (nine existing runs)
LEGACY_MANIFEST  = DRIVE_ROOT / 'final_results/casia_stage1_3modes_3seeds_v1/final_manifest.json'  # sha256 + earlier test results
E2E_ROOT         = DRIVE_ROOT / f'checkpoints/topic2_e2e_{EPOCHS}ep'    # new training runs
EVAL_DIR         = DRIVE_ROOT / f'reports/eval_{EPOCHS}ep'
TABLE_DIR        = DRIVE_ROOT / f'reports/tables_{EPOCHS}ep'

# Stage switches
EVAL_ONLY              = False   # True: no training, no GPU required, reuse saved thresholds (CPU runtime is fine)
RUN_TRAIN_UNET         = True
RUN_TRAIN_SEGFORMER    = True
RUN_CASIA_TEST         = True
RUN_COVERAGE           = True
COVERAGE_MASKS_CHECKED = False   # set to True only AFTER inspecting the Coverage mask preview in section 6
RUN_DEMO               = True

# Section 10: extra analyses that need no training
RUN_COLUMBIA             = True
COLUMBIA_SOURCE          = DRIVE_ROOT / 'datasets/Columbia'   # a folder or an archive (.zip / .tar.bz2) on Drive
COLUMBIA_MASKS_CHECKED   = False   # set to True only AFTER inspecting the Columbia mask preview
RUN_IMD2020              = True
IMD2020_SOURCE           = DRIVE_ROOT / 'datasets/IMD2020'   # folder or archive; one sub-folder per source photo
IMD2020_MASKS_CHECKED    = False
RUN_NIST16               = True
NIST16_SOURCE            = DRIVE_ROOT / 'datasets/NC2016_Test0613'   # folder or archive (NIST Nimble 2016)
NIST16_MASKS_CHECKED     = False   # NIST masks are inverted (black = manipulated): check the preview
RUN_ROBUSTNESS           = True
ROBUSTNESS_PERTURBATIONS = ['jpeg:90', 'jpeg:70', 'jpeg:50', 'resize:0.5', 'noise:5']
ROBUSTNESS_SEEDS         = [7]     # a subset of SEEDS keeps it fast; use SEEDS for every run (GPU recommended)
RUN_SCORE_STUDY          = True
SCORE_STUDY_REEVALUATE   = False   # True once: re-run val/test so per_image.csv holds the newer score methods
UPLOAD_OWN_IMAGE       = False   # True: section 9 shows an upload button (Run all waits for a file)

## 1. GPU, Google Drive and source code

In [ ]:
import os, sys, subprocess, shutil, json, zipfile
from pathlib import PurePosixPath

def run(cmd, cwd=None):
    """Run a command, stream its output, and stop the notebook on failure."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError(f'Command failed ({process.returncode}): {" ".join(cmd)}')

import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| Python', sys.version.split()[0])
else:
    assert EVAL_ONLY, 'Enable a GPU (Runtime → Change runtime type → GPU), or set EVAL_ONLY = True to evaluate on CPU'
    print('EVAL_ONLY on CPU | torch', torch.__version__, '| Python', sys.version.split()[0])

from google.colab import drive
drive.mount('/content/drive')
for p in (E2E_ROOT, EVAL_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

if (REPO / '.git').is_dir():
    run(['git', 'pull', '--ff-only'])
else:
    run(['git', 'clone', REPO_URL, REPO], cwd='/content')
run(['git', 'log', '--oneline', '-1'])

In [ ]:
# Install packages (keep Colab's GPU PyTorch; the project declares Python 3.11, so skip the version check)
run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch==0.6.0', 'gradio', 'pytest'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', '-e', '.'])
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
import segmentation_models_pytorch as smp
print('smp', smp.__version__)
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'])

## 2. CASIA v2 and the train / validation / test manifests

In [ ]:
def casia_ready(path):
    return all((path / d).is_dir() for d in ('Au', 'Tp', 'Groundtruth'))

LOCAL_CASIA.parent.mkdir(parents=True, exist_ok=True)
if not casia_ready(LOCAL_CASIA):
    if LOCAL_CASIA.is_symlink():
        LOCAL_CASIA.unlink()
    source = None
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        candidates = [downloaded] + [p for p in downloaded.rglob('*') if p.is_dir()]
        source = next((p for p in candidates if casia_ready(p)), None)
    except Exception as error:
        print('KaggleHub unavailable:', error)
    if source is not None:
        LOCAL_CASIA.symlink_to(source, target_is_directory=True)
        print('Using KaggleHub copy:', source)
    else:
        assert CASIA_ZIP.is_file(), f'No KaggleHub copy and {CASIA_ZIP} not found'
        if LOCAL_CASIA.exists():
            shutil.rmtree(LOCAL_CASIA)
        with zipfile.ZipFile(CASIA_ZIP) as archive:
            for item in archive.infolist():   # reject unsafe archive paths
                parts = PurePosixPath(item.filename).parts
                assert parts and parts[0] == 'CASIA2' and '..' not in parts and not item.filename.startswith('/'), item.filename
            archive.extractall(LOCAL_CASIA.parent)
        print('Extracted from Drive ZIP')
assert casia_ready(LOCAL_CASIA)
print('CASIA2:', LOCAL_CASIA.resolve())

In [ ]:
import pandas as pd
LOCAL_MANIFEST.mkdir(parents=True, exist_ok=True)
new_prefix = str(LOCAL_CASIA) + '/'
for name in ('train', 'val', 'test'):
    frame = pd.read_csv(MANIFEST_BACKUP / f'{name}.csv', keep_default_na=False)
    for column in ('image_path', 'mask_path'):
        def convert(path):
            if not path:
                return ''
            assert path.startswith(MANIFEST_PREFIX), f'Unexpected path: {path}'
            return new_prefix + path[len(MANIFEST_PREFIX):]
        frame[column] = frame[column].map(convert)
    missing = [p for p in list(frame['image_path']) + [m for m in frame['mask_path'] if m] if not Path(p).is_file()]
    assert not missing, f'{name}: {len(missing)} files missing, e.g. {missing[:2]}'
    frame.to_csv(LOCAL_MANIFEST / f'{name}.csv', index=False)
    print(f'{name.upper():5s} {len(frame):5d} images | {frame["label"].value_counts().to_dict()}')
TRAIN, VAL, TEST = (LOCAL_MANIFEST / f'{s}.csv' for s in ('train', 'val', 'test'))

In [ ]:
# Checks: the current split must be IDENTICAL to the split used for the existing checkpoints,
# and train / validation / test must not overlap.
import hashlib
from forgery_localization.datasets.casia_dataset import CASIADataset, validate_split_isolation

def legacy_split_signature(dataset):
    fields = ['image_hash', 'label', 'source_group', 'mask_verified', 'mask_source', 'mask_threshold', 'threshold_approved']
    rows = [{**{k: str(r.get(k, '')) for k in fields},
             'mask_name': Path(r['mask_path']).name if r['mask_path'] is not None else ''} for r in dataset.records]
    return hashlib.sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()

train_ds = CASIADataset(TRAIN, root='/', image_size=(256, 256))
val_ds = CASIADataset(VAL, root='/', image_size=(256, 256))
test_ds = CASIADataset(TEST, root='/', image_size=(256, 256))
validate_split_isolation(train_ds, val_ds, test_ds)
print('No group / path / hash overlap between train, validation and test: PASS')

legacy_configs = sorted(LEGACY_UNET_ROOT.glob('*_seed*/config.json'))
if legacy_configs:
    reference = json.loads(legacy_configs[0].read_text())
    assert legacy_split_signature(train_ds) == reference['train_signature'], 'Train split DIFFERS from the existing checkpoints!'
    assert legacy_split_signature(val_ds) == reference['val_signature'], 'Validation split DIFFERS from the existing checkpoints!'
    print('Split identical to the existing checkpoints: PASS')
else:
    print('No existing checkpoints found — every run will be trained from scratch.')
del train_ds, val_ds   # keep test_ds for the prediction section

## 3. Training

* **U-Net–ResNet34** × {RGB, RGB+Noise, RGB+DCT} × 3 seeds = 9 runs. Existing runs with the same epoch budget are reused; missing runs are trained with `scripts/train_multiinput.py` (same protocol: weighted BCE + Dice, seed-deterministic flips, checkpoint selection by validation micro Dice@0.5).
* **SegFormer** (Transformer) under the same protocol, for the CNN vs Transformer comparison.

In [ ]:
def run_complete(directory, epochs=EPOCHS):
    # Must equal the protocol's epoch budget (never mix 10-epoch runs into a 30-epoch study)
    history = directory / 'history.csv'
    return (directory / 'best_dice.pt').is_file() and history.is_file() and \
        int(pd.read_csv(history)['epoch'].max()) == epochs

def checkpoint_for(arch, mode, seed):
    """Path of a completed best_dice.pt, preferring the existing notebook run; None if not available."""
    if arch == 'unet_resnet34':
        legacy = LEGACY_UNET_ROOT / f'{mode}_seed{seed}'
        if run_complete(legacy):
            return legacy / 'best_dice.pt'
    new = E2E_ROOT / f'{arch}_{mode}_seed{seed}'
    return new / 'best_dice.pt' if run_complete(new) else None

PLAN = [('unet_resnet34', m, s) for s in SEEDS for m in UNET_MODES] + \
       [(SEGFORMER_ARCH, m, s) for s in SEEDS for m in SEGFORMER_MODES]

def show_status():
    rows = [dict(architecture=a, mode=m, seed=s, checkpoint=str(checkpoint_for(a, m, s) or '— not available'))
            for a, m, s in PLAN]
    display(pd.DataFrame(rows))
show_status()

In [ ]:
# Integrity check of the nine existing checkpoints: sha256 must match final_manifest.json (frozen on 2026-11-17)
import hashlib
LEGACY_RESULTS = {}
if LEGACY_MANIFEST.is_file():
    frozen = json.loads(LEGACY_MANIFEST.read_text())
    expected = {f['path']: f['sha256'] for f in frozen['files']}
    LEGACY_RESULTS = {f"{c['mode']}_seed{c['seed']}": c for c in frozen['checkpoints']}
    rows = []
    for key in LEGACY_RESULTS:
        path = LEGACY_UNET_ROOT / key / 'best_dice.pt'
        digest = hashlib.sha256()
        with open(path, 'rb') as stream:
            for block in iter(lambda: stream.read(1 << 22), b''):
                digest.update(block)
        rows.append(dict(run=key, sha256_ok=digest.hexdigest() == expected[f'checkpoints/{key}/best_dice.pt'],
                         selected_epoch=LEGACY_RESULTS[key]['selected_epoch']))
    integrity = pd.DataFrame(rows); display(integrity)
    assert integrity['sha256_ok'].all(), 'An existing checkpoint differs from the frozen version!'
else:
    print(LEGACY_MANIFEST, 'not found — skipping the sha256 check')

In [ ]:
def train(arch, mode, seed):
    if checkpoint_for(arch, mode, seed) is not None:
        print(f'SKIP {arch} {mode} seed {seed}: already complete'); return
    if EVAL_ONLY:
        print(f'EVAL_ONLY: {arch} {mode} seed {seed} is not trained and will be left out'); return
    run([sys.executable, 'scripts/train_multiinput.py', '--architecture', arch, '--mode', mode, '--seed', seed,
         '--train-manifest', TRAIN, '--val-manifest', VAL, '--root', '/', '--output-root', E2E_ROOT,
         '--epochs', EPOCHS])

if RUN_TRAIN_UNET:
    for seed in SEEDS:
        for mode in UNET_MODES:
            train('unet_resnet34', mode, seed)

In [ ]:
if RUN_TRAIN_SEGFORMER:
    for seed in SEEDS:
        for mode in SEGFORMER_MODES:
            train(SEGFORMER_ARCH, mode, seed)
show_status()

In [ ]:
# Validation learning curves of all completed runs
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for arch, mode, seed in PLAN:
    ckpt = checkpoint_for(arch, mode, seed)
    if ckpt is None:
        continue
    history = pd.read_csv(ckpt.parent / 'history.csv')
    label = f'{arch.split("_")[0]}-{mode}-s{seed}'
    axes[0].plot(history['epoch'], history['val_micro_dice'], label=label)
    axes[1].plot(history['epoch'], history['val_loss'], label=label)
axes[0].set_title('Validation micro Dice @0.5'); axes[1].set_title('Validation loss')
for ax in axes: ax.set_xlabel('epoch'); ax.grid(alpha=.3)
axes[0].legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

## 4. Calibrate the Real/Fake threshold on VALIDATION

An image is **Forged** when its `image_score` (mean of the 1% most suspicious pixels) is at least the threshold. The threshold maximizing F1 on validation is saved as `thresholds.json` next to each checkpoint and then **frozen** for every test set.

In [ ]:
CHECKPOINTS = [str(c) for c in (checkpoint_for(a, m, s) for a, m, s in PLAN) if c is not None]
print(len(CHECKPOINTS), 'completed checkpoints'); print(*CHECKPOINTS, sep='\n')
assert CHECKPOINTS, 'No completed checkpoints'

def evaluate(manifest, dataset_name, extra=(), checkpoints=None):
    run([sys.executable, 'scripts/evaluate.py', '--checkpoint', *(checkpoints or CHECKPOINTS), '--manifest', manifest,
         '--root', '/', '--dataset-name', dataset_name, '--output-dir', EVAL_DIR, *extra])

# EVAL_ONLY reuses the saved thresholds when every checkpoint has one and its validation summary exists
missing_thresholds = [c for c in CHECKPOINTS if not (Path(c).parent / 'thresholds.json').is_file()]
val_summaries = list((EVAL_DIR / 'casia_val').glob('*/summary.json'))
if EVAL_ONLY and not missing_thresholds and len(val_summaries) >= len(CHECKPOINTS):
    print(f'EVAL_ONLY: reusing saved thresholds.json and {len(val_summaries)} validation summaries')
else:
    evaluate(VAL, 'casia_val', ['--calibrate'])

## 5. CASIA v2 test set

In [ ]:
if RUN_CASIA_TEST:
    evaluate(TEST, 'casia_test', ['--isolation-manifests', TRAIN, VAL, '--save-visuals', '8'])

In [ ]:
# Cross-check: the new code (fp32) should reproduce the test micro Dice of the earlier report (fp16 autocast);
# small differences are expected.
rows = []
for key, old in LEGACY_RESULTS.items():
    summary_path = EVAL_DIR / 'casia_test' / f'unet_resnet34_{key}' / 'summary.json'
    if summary_path.is_file():
        new = json.loads(summary_path.read_text())['pixel']
        rows.append(dict(run=key, old_micro_dice=old['test_micro_dice'], new_micro_dice=new['micro_dice'],
                         diff=new['micro_dice'] - old['test_micro_dice'],
                         old_mean_tampered_dice=old['test_mean_tampered_dice'], new_mean_tampered_dice=new['mean_tampered_dice']))
if rows:
    check = pd.DataFrame(rows); display(check.round(4))
    worst = check['diff'].abs().max()
    print('PASS' if worst < 0.01 else 'WARNING: difference > 0.01 — check preprocessing', f'(max |diff| = {worst:.4f})')

## 6. Cross-dataset: Coverage (zero-shot — no fine-tuning, no threshold changes)

COVERAGE is no longer stored in its GitHub repository: download the full folder from the authors' OneDrive link (README of https://github.com/wenbihan/coverage) and upload the ZIP to `COVERAGE_ZIP` on Drive. Nine forged images whose masks do not match the image size are skipped automatically (191 images remain).

The first run only builds the manifest and a **mask preview**. Check that the red area covers the copy-move region. If it does, set `COVERAGE_MASKS_CHECKED = True` in the CONFIG cell and re-run from here.

In [ ]:
COVERAGE_CSV = Path('/content/manifests/coverage_test.csv')
COVERAGE_PREVIEW = Path('/content/coverage_preview.png')

def find_coverage_root(base):
    for zip_path in base.rglob('*.zip'):
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(zip_path.parent)
    for candidate in [base, *[p for p in base.rglob('*') if p.is_dir()]]:
        if (candidate / 'image').is_dir() and (candidate / 'mask').is_dir():
            return candidate
    raise FileNotFoundError(f'No folder containing image/ and mask/ under {base}')

if RUN_COVERAGE:
    assert COVERAGE_ZIP.is_file(), (f'{COVERAGE_ZIP} not found: download COVERAGE from the OneDrive link in '
                                    'https://github.com/wenbihan/coverage and upload the ZIP there (or set RUN_COVERAGE = False)')
    coverage_base = Path('/content/coverage_official')
    if not coverage_base.exists():
        coverage_base.mkdir(parents=True)
        shutil.copy(COVERAGE_ZIP, coverage_base)
    COVERAGE_ROOT = find_coverage_root(coverage_base)
    print('Coverage root:', COVERAGE_ROOT)
    command = [sys.executable, 'scripts/build_external_manifest.py', 'coverage', '--dataset-root', COVERAGE_ROOT,
               '--output', COVERAGE_CSV, '--preview', COVERAGE_PREVIEW]
    if COVERAGE_MASKS_CHECKED:
        command.append('--masks-checked')
    run(command)
    from IPython.display import Image as IPImage, display
    display(IPImage(str(COVERAGE_PREVIEW)))

In [ ]:
if RUN_COVERAGE and COVERAGE_MASKS_CHECKED:
    evaluate(COVERAGE_CSV, 'coverage', ['--save-visuals', '6'])
elif RUN_COVERAGE:
    print('Coverage not evaluated yet: inspect the preview above, then set COVERAGE_MASKS_CHECKED = True.')

## 7. Result tables for the report

In [ ]:
run([sys.executable, 'scripts/summarize_results.py', '--eval-dir', EVAL_DIR, '--output-dir', TABLE_DIR])
from IPython.display import Markdown, display
display(Markdown('### Mean ± SD per configuration\n' + (TABLE_DIR / 'mean_sd.md').read_text(encoding='utf-8')))
display(Markdown('### Generalization (trained on CASIA → tested on each dataset)\n' + (TABLE_DIR / 'generalization.md').read_text(encoding='utf-8')))
paired = TABLE_DIR / 'paired_vs_rgb.csv'
if paired.exists():
    display(Markdown('### Same-seed difference to RGB (Noise/DCT − RGB)'))
    frame = pd.read_csv(paired)
    display(frame[['dataset', 'architecture', 'mode', 'seed', 'delta_pixel_micro_dice',
                   'delta_pixel_authentic_fp_pixel_pct', 'delta_image_f1', 'delta_image_auc']].round(4))

In [ ]:
# Image-level confusion matrix on the CASIA test set for every run
rows = []
for summary_path in sorted((EVAL_DIR / 'casia_test').glob('*/summary.json')):
    s = json.loads(summary_path.read_text())
    i = s['image']
    rows.append(dict(run=s['run'], TN=i['tn'], FP=i['fp'], FN=i['fn'], TP=i['tp'],
                     accuracy=i['accuracy'], precision=i['precision'], recall=i['recall'], f1=i['f1'], auc=i['auc'],
                     threshold=s['thresholds']['image_threshold']))
display(pd.DataFrame(rows).round(4))

## 8. Success and failure examples (for the error analysis)

In [ ]:
from IPython.display import Image as IPImage, display
# The best model is selected by VALIDATION micro Dice (never by test)
val_scores = []
for summary_path in (EVAL_DIR / 'casia_val').glob('*/summary.json'):
    s = json.loads(summary_path.read_text())
    val_scores.append((s['pixel']['micro_dice'], s['run'], s['checkpoint']))
val_scores.sort(reverse=True)
BEST_VAL_DICE, BEST_RUN, BEST_CHECKPOINT = val_scores[0]
print(f'Best model by validation: {BEST_RUN} (validation micro Dice {BEST_VAL_DICE:.4f})')
print('Panels: image | ground truth | probability map | prediction')
visuals = EVAL_DIR / 'casia_test' / BEST_RUN / 'visuals'
for group in ('best_tampered', 'worst_tampered', 'worst_authentic_fp'):
    files = sorted((visuals / group).glob('*.png'))[:3]
    print(f'\n== {group} ({len(files)}) ==')
    for f in files:
        print(f.name); display(IPImage(str(f), width=900))

## 9. Single-image prediction: Real/Fake, confidence, mask, bounding boxes, suspicious area

By default a tampered image from the CASIA test set is used; set `UPLOAD_OWN_IMAGE = True` to upload your own image.

In [ ]:
from PIL import Image
from forgery_localization.inference import Predictor, load_thresholds, render_overlay, report_text

thresholds_path = Path(BEST_CHECKPOINT).parent / 'thresholds.json'
predictor = Predictor(BEST_CHECKPOINT, thresholds=load_thresholds(thresholds_path))

image_paths = []
if UPLOAD_OWN_IMAGE:
    from google.colab import files
    image_paths = [Path(name) for name in files.upload()]
if not image_paths:
    image_paths = [next(r['image_path'] for r in test_ds.records if r['label'] == 'tampered')]

for path in image_paths:
    with Image.open(path) as opened:
        image = opened.convert('RGB')
    probabilities, mask, summary = predictor.predict(image)
    display(Markdown(f'**{Path(path).name}**\n\n' + report_text(summary, predictor.name)))
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image); axes[0].set_title('Input image')
    axes[1].imshow(probabilities, cmap='magma', vmin=0, vmax=1); axes[1].set_title('Probability map')
    axes[2].imshow(render_overlay(image, mask, summary)); axes[2].set_title('Mask + bounding boxes')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

## 10. Extra analyses (no training)

Three questions the main results leave open, answered with the frozen checkpoints and thresholds:

* **10a. Columbia (splicing, zero-shot)** — does the COVERAGE failure come from *copy-move* or from *any unseen dataset*? Columbia Uncompressed is a splicing dataset (183 authentic, 180 spliced TIFF images) like CASIA, but from other cameras.
* **10b. Robustness** — how much do JPEG re-compression, downscaling and noise reduce performance, and are the Noise/DCT inputs more fragile than RGB?
* **10c. Real/Fake score methods** — is there an image score that is more stable across seeds than the mean of the top 1% pixels? Thresholds and the recommended method are chosen on validation only.

**Columbia data:** download *Columbia Uncompressed Image Splicing Detection Evaluation Dataset* from https://www.ee.columbia.edu/ln/dvmm/downloads/authsplcuncmp/ (accept the terms of use), then upload the `4cam_auth` and `4cam_splc` archives or folders to `COLUMBIA_SOURCE` on Drive. As for COVERAGE, the first run only writes a mask preview: check that red covers the spliced object, then set `COLUMBIA_MASKS_CHECKED = True` and re-run from here. If red covers the *background* instead, add `'--columbia-region', 'green'` to the command below.


In [ ]:
# 10a. Columbia: manifest, mask preview, then zero-shot evaluation with the CASIA thresholds
COLUMBIA_CSV = Path('/content/manifests/columbia_test.csv')
COLUMBIA_PREVIEW = Path('/content/columbia_preview.png')
COLUMBIA_LOCAL = Path('/content/columbia')
ARCHIVES = ('.zip', '.tar', '.tar.bz2', '.tbz2', '.tar.gz', '.tgz')

def unpack_all(folder):
    # Unpack every archive below `folder` (repeat for archives inside archives)
    done = set()
    while True:
        pending = [p for p in folder.rglob('*') if p.is_file() and p.name.lower().endswith(ARCHIVES) and p not in done]
        if not pending:
            return
        for archive in pending:
            shutil.unpack_archive(str(archive), str(archive.parent))
            done.add(archive)

if RUN_COLUMBIA and not COLUMBIA_SOURCE.exists():
    print(f'Columbia skipped: {COLUMBIA_SOURCE} not found (see the instructions above).')
elif RUN_COLUMBIA:
    if not COLUMBIA_LOCAL.exists():
        if COLUMBIA_SOURCE.is_dir():
            shutil.copytree(COLUMBIA_SOURCE, COLUMBIA_LOCAL)
        else:
            COLUMBIA_LOCAL.mkdir(parents=True)
            shutil.copy(COLUMBIA_SOURCE, COLUMBIA_LOCAL)
        unpack_all(COLUMBIA_LOCAL)
    command = [sys.executable, 'scripts/build_external_manifest.py', 'columbia', '--dataset-root', COLUMBIA_LOCAL,
               '--output', COLUMBIA_CSV, '--preview', COLUMBIA_PREVIEW]
    if COLUMBIA_MASKS_CHECKED:
        command.append('--masks-checked')
    run(command)
    from IPython.display import Image as IPImage, display
    if COLUMBIA_PREVIEW.is_file():
        display(IPImage(str(COLUMBIA_PREVIEW)))
    if COLUMBIA_MASKS_CHECKED:
        evaluate(COLUMBIA_CSV, 'columbia', ['--save-visuals', '6'])
    else:
        print('Columbia not evaluated yet: inspect the preview, then set COLUMBIA_MASKS_CHECKED = True.')


**10a (cont.). IMD2020 and NIST16 (zero-shot)** complete the five datasets listed in topic 2 (CASIA v2 + Columbia + Coverage + IMD2020 + NIST16).

* **IMD2020** (real-life manipulations): download the *IMD2020 real-life manipulated images* set from the authors' page (http://staff.utia.cas.cz/novozada/db/) and upload the folder or archive to `IMD2020_SOURCE`. Layout: one sub-folder per photo with `{id}_orig.*`, `{id}_{k}.*` and `{id}_{k}_mask.*`.
* **NIST16** (NC2016_Test0613): obtain it from NIST (registration required) and upload it to `NIST16_SOURCE`. Rows come from `reference/manipulation/NC2016-manipulation-ref.csv`. NIST masks mark the manipulated region in **black**; the builder inverts them. If the preview shows red on the untouched area, add `'--nist-manipulated', 'white'` to the command.

As for the other sets: inspect each preview first, then set the matching `*_MASKS_CHECKED = True` and re-run.

In [ ]:
# 10a (cont.). IMD2020 and NIST16: manifest, mask preview, then zero-shot evaluation with the CASIA thresholds
from IPython.display import Image as IPImage, display

def external_set(name, run_flag, source, checked, extra_args=()):
    if not run_flag:
        return None
    if not source.exists():
        print(f'{name} skipped: {source} not found (see the instructions above).')
        return None
    local = Path(f'/content/{name}')
    if not local.exists():
        if source.is_dir():
            shutil.copytree(source, local)
        else:
            local.mkdir(parents=True)
            shutil.copy(source, local)
        unpack_all(local)
    # Locate the dataset root inside the unpacked copy
    if name == 'nist16':
        refs = sorted(local.rglob('NC2016-manipulation-ref.csv'))
        root = refs[0].parents[2] if refs else local          # <root>/reference/manipulation/<csv>
    else:
        origs = sorted(local.rglob('*_orig.*'))
        root = origs[0].parents[1] if origs else local        # <root>/<photo id>/<id>_orig.jpg
    csv_path = Path(f'/content/manifests/{name}_test.csv')
    preview = Path(f'/content/{name}_preview.png')
    command = [sys.executable, 'scripts/build_external_manifest.py', name, '--dataset-root', root,
               '--output', csv_path, '--preview', preview, *extra_args]
    if checked:
        command.append('--masks-checked')
    run(command)
    if preview.is_file():
        display(IPImage(str(preview)))
    if checked:
        evaluate(csv_path, name, ['--save-visuals', '6'])
    else:
        print(f'{name} not evaluated yet: inspect the preview, then set {name.upper()}_MASKS_CHECKED = True.')
    return csv_path

IMD2020_CSV = external_set('imd2020', RUN_IMD2020, IMD2020_SOURCE, IMD2020_MASKS_CHECKED)
NIST16_CSV = external_set('nist16', RUN_NIST16, NIST16_SOURCE, NIST16_MASKS_CHECKED)

In [ ]:
# 10b. Robustness on the CASIA test set: perturb the image, keep the mask and the validation thresholds
from forgery_localization.perturbations import perturbation_tag

if RUN_ROBUSTNESS:
    robust_checkpoints = [str(c) for c in (checkpoint_for(a, m, s) for a, m, s in PLAN if s in ROBUSTNESS_SEEDS)
                          if c is not None]
    for spec in ROBUSTNESS_PERTURBATIONS:
        name = f'casia_test_{perturbation_tag(spec)}'
        existing = list((EVAL_DIR / name).glob('*/summary.json'))
        if len(existing) >= len(robust_checkpoints):
            print(f'SKIP {name}: already evaluated ({len(existing)} runs)'); continue
        evaluate(TEST, name, ['--perturb', spec], checkpoints=robust_checkpoints)

    rows = []
    for spec in ['none', *ROBUSTNESS_PERTURBATIONS]:
        name = 'casia_test' if spec == 'none' else f'casia_test_{perturbation_tag(spec)}'
        for summary_path in sorted((EVAL_DIR / name).glob('*/summary.json')):
            s = json.loads(summary_path.read_text())
            if s['seed'] not in ROBUSTNESS_SEEDS:
                continue
            rows.append(dict(perturbation=spec, architecture=s['architecture'], mode=s['mode'], seed=s['seed'],
                             micro_dice=s['pixel']['micro_dice'], image_f1=s['image']['f1'], image_auc=s['image']['auc']))
    robust = pd.DataFrame(rows)
    clean = robust[robust.perturbation == 'none'].set_index(['architecture', 'mode', 'seed'])
    for metric in ('micro_dice', 'image_auc'):
        table = robust.pivot_table(index=['architecture', 'mode'], columns='perturbation', values=metric, aggfunc='mean')
        table = table[['none', *ROBUSTNESS_PERTURBATIONS]]
        display(Markdown(f'### {metric} under perturbation (mean over seeds {ROBUSTNESS_SEEDS})'))
        display(table.round(3))
        display(Markdown(f'Relative change vs clean ({metric})'))
        display((table.sub(table['none'], axis=0).div(table['none'], axis=0) * 100).round(1).astype(str) + '%')
    robust.to_csv(TABLE_DIR / 'robustness.csv', index=False)
    print('Saved', TABLE_DIR / 'robustness.csv')


In [ ]:
# 10c. Real/Fake score methods: per method, threshold chosen on validation, applied unchanged to each test set
if RUN_SCORE_STUDY:
    if SCORE_STUDY_REEVALUATE:
        # per_image.csv files written before the newer score methods existed hold only three of them.
        # Re-calibrating with the default method rewrites identical thresholds.json files.
        evaluate(VAL, 'casia_val', ['--calibrate'])
        evaluate(TEST, 'casia_test')
        if RUN_COVERAGE and COVERAGE_MASKS_CHECKED:
            evaluate(COVERAGE_CSV, 'coverage')
    SCORE_DIR = DRIVE_ROOT / f'reports/score_study_{EPOCHS}ep'
    targets = [d for d in ('casia_test', 'coverage', 'columbia', 'imd2020', 'nist16') if (EVAL_DIR / d).is_dir()]
    run([sys.executable, 'scripts/score_study.py', '--eval-dir', EVAL_DIR, '--calibration', 'casia_val',
         '--datasets', *targets, '--output-dir', SCORE_DIR])


## 11. Web demo (Gradio, public link)
Pick a model from the dropdown to compare RGB / Noise / DCT / SegFormer on the same image.

In [ ]:
if RUN_DEMO:
    import importlib.util
    spec = importlib.util.spec_from_file_location('demo_app', REPO / 'app' / 'demo.py')
    demo_app = importlib.util.module_from_spec(spec); spec.loader.exec_module(demo_app)
    seed0 = SEEDS[0]
    demo_checkpoints = [str(c) for c in (checkpoint_for(a, m, seed0) for a, m, s in PLAN if s == seed0) if c is not None]
    demo = demo_app.build_interface(demo_app.load_predictors(demo_checkpoints, 'auto'))
    demo.launch(share=True, debug=False)